# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline (image and audio generation not working as for now)

In [ ]:
# imports

from dotenv import load_dotenv
from IPython.display import Markdown, display
from datetime import datetime
import time
import json
from revealer import reveal
from openai import OpenAI
import os
import gradio as gr
import sqlite3

load_dotenv(override=True)

In [ ]:
load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key and openrouter_api_key.startswith("sk-or-"):
    print("OpenRouter API Key looks good so far")
else:
    print("OpenRouter API Key doesn't seem right")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")
    
MODEL = "gemini-3.5-flash-lite"
MODEL_OPEN_ROUTER = "openai/gpt-5-nano"

openrouter_base_url = "https://openrouter.ai/api/v1"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

openai = OpenAI(api_key=google_api_key, base_url=gemini_url)
openrouter = OpenAI(base_url=openrouter_base_url, api_key=openrouter_api_key)

DB = "prices.db"

In [134]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [135]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [136]:
def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: Setting price for {city} price for {price}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [137]:
get_price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "price": {
              "type": "number",
              "description": "The price of the ticket"
            }
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

tools: list[dict[str, str | dict[str, str | dict[str, bool | list[str] | str | dict[str, dict[str, str]]]]]] = [{"type": "function", "function": get_price_function}, {"type": "function", "function": set_price_function}]

In [138]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [139]:
available_functions = {
  'get_ticket_price': get_ticket_price, 
  'set_ticket_price': set_ticket_price
}

In [140]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        
        function_name = tool_call.function.name
        function_to_call = available_functions.get(function_name)

        if not function_to_call:
             responses.append({
                "role": "tool",
                "content": f'Error: Function {function_name} not available',
                "tool_call_id": tool_call.id
            })

        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')


        if tool_call.function.name == "set_ticket_price":
            price = arguments.get('price')
            price_details = function_to_call(city, price)
            responses.append({
                "role": "tool",
                "content": f"Ticket price {price} for this city {city} updated successfully",
                "tool_call_id": tool_call.id
            })

        if tool_call.function.name == "get_ticket_price":
            city = arguments.get('destination_city')
            price_details = function_to_call(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
            
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## A bit more about what Gradio actually does:

1. Gradio constructs a frontend Svelte app based on our Python description of the UI
2. Gradio starts a server built upon the Starlette web framework listening on a free port that serves this Svelte app
3. Gradio creates backend routes for our callbacks, like chat(), which calls our functions

# Let's go multi-modal!!

We can use gpt-image-1-mini, the image generation model behind GPT-5, to make us some images

In [ ]:
import base64
from io import BytesIO
from PIL import Image
import urllib
from google import genai
from google.genai import types

In [142]:
client = genai.Client(
    api_key=os.environ.get("GOOGLE_API_KEY"),
    http_options=types.HttpOptions(api_version="v1beta")
)

In [ ]:
def artist(city, effort=None):
    try:
        # Use the dedicated image generation method and model
        response = client.models.generate_content(
            model='gemini-3.8-flash', 
            contents=f"An image representing a vacation in {city}...",
            config=types.GenerateContentConfig(
                response_modalities=['IMAGE']
            )
        )

        # Extract image bytes from the native gemini parts payload
        for part in response.candidates[0].content.parts:
            if part.inline_data:
                # The SDK decodes base64 string automatically into raw bytes
                return part.inline_data.data 
                
    except Exception as e:
        print(f"Generation failed: {e}")
        return None
        
    return None

In [ ]:
image_bytes = artist("New York City")


if image_bytes:
    # 2. Convert raw bytes into a PIL Image object so Jupyter can render it
    image = Image.open(BytesIO(image_bytes))
    display(image)
else:
    print("Could not retrieve image.")


In [ ]:
def talker(message):
    response = openai.audio.speech.create(
      model=MODEL,
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

## Let's bring this home:

1. A multi-modal AI assistant with image and audio generation
2. Tool callling with database lookup
3. A step towards an Agentic workflow


In [81]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    # voice = talker(reply)

    if cities:
        svg_string = artist(cities[0], 'low')
        # URL-encode the SVG text and prefix with the data schema
        encoded_svg = urllib.parse.quote(svg_string)
        image = f"data:image/svg+xml,{encoded_svg}"
    
    updated_history = history + [{"role": "assistant", "content": "Here is your image!"}]
    
    return updated_history, image
    

In [82]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## The 3 types of Gradio UI

`gr.Interface` is for standard, simple UIs

`gr.ChatInterface` is for standard ChatBot UIs

`gr.Blocks` is for custom UIs where you control the components and the callbacks

In [ ]:
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    # with gr.Row():
    #     audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks
    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, image_output]
    )

ui.launch(inbrowser=True, auth=("k", "v"))